# Part 3: Damned if you do, damned if you don't (R Implementation)

This notebook implements the "damned if you do, damned if you don't" example from Lab7 and extends it with additional analysis using R.

In [ ]:
# Load required libraries
library(ggplot2)
library(dplyr)
library(dagitty)
library(ggdag)
library(broom)
library(gridExtra)
library(tidyr)
library(xtable)

# Set random seed for reproducibility
set.seed(42)

# Create output directory if it doesn't exist
output_dir <- "../output"
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

## Original "Damned if you do, damned if you don't" Example

From Lab7 Example 4: This represents a situation where it's unclear whether Z should be used as a control.

In [ ]:
# Create the original DAG from Lab7 Example 4
original_dag <- dagify(
  Y ~ X + Z,
  coords = list(
    x = c(X = 0, Z = 1, Y = 2),
    y = c(X = 0, Z = 1, Y = 0)
  )
)

p_orig <- ggdag(original_dag) + 
  theme_dag() + 
  ggtitle("Original DAG: Damned if you do, damned if you don't")

print(p_orig)
ggsave(file.path(output_dir, "part3_original_dag_R.png"), p_orig, width = 8, height = 6, dpi = 300)

In [ ]:
# Simulate data for original example (following Lab7 structure)
n <- 10000

# Generate data as in Lab7
Z_orig <- rnorm(n, 0, 1)
X_orig <- rnorm(n, 0, 1)  # X independent of Z
Y_orig <- X_orig + Z_orig + rnorm(n, 0, 1)  # Both X and Z affect Y

# True causal effect of X on Y is 1.0
true_effect <- 1.0

cat(sprintf("Original example data generated. True causal effect of X on Y: %.1f\n", true_effect))
cat(sprintf("Sample size: %d\n", n))

In [ ]:
# Run regressions with and without Z control

# Regression 1: Y vs X (without Z)
model1_orig <- lm(Y_orig ~ X_orig)
coef1_orig <- coef(model1_orig)["X_orig"]
ci1_orig <- confint(model1_orig, "X_orig", level = 0.99)

# Regression 2: Y vs X, Z (with Z)
model2_orig <- lm(Y_orig ~ X_orig + Z_orig)
coef2_orig <- coef(model2_orig)["X_orig"]
ci2_orig <- confint(model2_orig, "X_orig", level = 0.99)

cat("Original Example Results:\n")
cat(sprintf("Without Z control: %.4f [99%% CI: %.4f, %.4f]\n", coef1_orig, ci1_orig[1], ci1_orig[2]))
cat(sprintf("With Z control:    %.4f [99%% CI: %.4f, %.4f]\n", coef2_orig, ci2_orig[1], ci2_orig[2]))
cat(sprintf("True effect:       %.4f\n", true_effect))

In [ ]:
# Plot coefficients for original example
orig_results <- data.frame(
  Model = c("Without Z", "With Z"),
  Coefficient = c(coef1_orig, coef2_orig),
  CI_Lower = c(ci1_orig[1], ci2_orig[1]),
  CI_Upper = c(ci1_orig[2], ci2_orig[2])
)

p_orig_coef <- ggplot(orig_results, aes(x = Model, y = Coefficient)) +
  geom_point(size = 4) +
  geom_errorbar(aes(ymin = CI_Lower, ymax = CI_Upper), 
                width = 0.2, size = 1) +
  geom_hline(yintercept = true_effect, color = "red", linetype = "dashed", size = 1) +
  labs(
    x = "Model Specification",
    y = "Estimated Effect of X on Y",
    title = "Original Example: Effect Estimates with 99% Confidence Intervals"
  ) +
  theme_minimal() +
  annotate("text", x = 1, y = true_effect + 0.05, 
           label = paste("True Effect =", true_effect), color = "red")

print(p_orig_coef)
ggsave(file.path(output_dir, "part3_original_coefficients_R.png"), p_orig_coef, 
       width = 10, height = 6, dpi = 300)

## Modified Example: Z also affects X

Now we modify the DAG so that Z also has an effect on X, and we can observe additional variables U1 and U2.

In [ ]:
# Create modified DAG where Z affects X, and add U1 and U2
modified_dag <- dagify(
  Y ~ X + U1 + U2 + Z,
  X ~ U1 + Z,
  Z ~ U2,
  coords = list(
    x = c(U1 = 0, U2 = 0, Z = 1, X = 2, Y = 3),
    y = c(U1 = 0, U2 = 2, Z = 1, X = 0.5, Y = 1)
  )
)

p_mod <- ggdag(modified_dag) + 
  theme_dag() + 
  ggtitle("Modified DAG: Z affects X, with observable U1 and U2")

print(p_mod)
ggsave(file.path(output_dir, "part3_modified_dag_R.png"), p_mod, width = 10, height = 8, dpi = 300)

In [ ]:
# Simulate data for modified example
n <- 10000

# Generate exogenous variables
U1 <- rnorm(n, 0, 1)
U2 <- rnorm(n, 0, 1)
eps_Z <- rnorm(n, 0, 1)
eps_X <- rnorm(n, 0, 1)
eps_Y <- rnorm(n, 0, 1)

# Generate endogenous variables following DAG structure
Z <- U2 + eps_Z          # U2 -> Z
X <- U1 + Z + eps_X      # U1 -> X, Z -> X  
Y <- X + U1 + U2 + Z + eps_Y  # X -> Y, U1 -> Y, U2 -> Y, Z -> Y

# Create DataFrame
df_modified <- data.frame(
  U1 = U1,
  U2 = U2,
  Z = Z,
  X = X,
  Y = Y
)

cat("Modified example data generated.\n")
cat(sprintf("True causal effect of X on Y: %.1f\n", true_effect))
cat(sprintf("Sample size: %d\n", n))
cat("\nData summary:\n")
print(summary(df_modified))

## Comprehensive Regression Analysis

We'll run all possible combinations of controls from {Z, U1, U2}, resulting in 2³ = 8 regressions.

In [ ]:
# Generate all possible combinations of controls
controls <- c("Z", "U1", "U2")

# Generate all subsets (power set)
generate_combinations <- function(items) {
  n <- length(items)
  combinations <- list()
  
  for (i in 0:(2^n - 1)) {
    subset <- c()
    for (j in 1:n) {
      if (bitwAnd(i, 2^(j-1)) != 0) {
        subset <- c(subset, items[j])
      }
    }
    combinations[[i + 1]] <- subset
  }
  
  return(combinations)
}

all_combinations <- generate_combinations(controls)

cat("All combinations of controls:\n")
for (i in seq_along(all_combinations)) {
  if (length(all_combinations[[i]]) == 0) {
    cat(sprintf("%d: None\n", i))
  } else {
    cat(sprintf("%d: %s\n", i, paste(all_combinations[[i]], collapse = ", ")))
  }
}
cat(sprintf("Total number of regressions: %d\n", length(all_combinations)))

In [ ]:
# Run all regressions
results_modified <- data.frame(
  Controls = character(length(all_combinations)),
  Beta = numeric(length(all_combinations)),
  SE = numeric(length(all_combinations)),
  Bias = numeric(length(all_combinations)),
  stringsAsFactors = FALSE
)

for (i in seq_along(all_combinations)) {
  controls_subset <- all_combinations[[i]]
  
  # Prepare regression formula
  if (length(controls_subset) == 0) {
    formula_str <- "Y ~ X"
    control_names <- "None"
  } else {
    formula_str <- paste("Y ~ X +", paste(controls_subset, collapse = " + "))
    control_names <- paste(controls_subset, collapse = ", ")
  }
  
  # Run regression
  model <- lm(as.formula(formula_str), data = df_modified)
  
  # Extract results for X
  coef_X <- coef(model)["X"]
  se_X <- summary(model)$coefficients["X", "Std. Error"]
  
  results_modified[i, "Controls"] <- control_names
  results_modified[i, "Beta"] <- coef_X
  results_modified[i, "SE"] <- se_X
  results_modified[i, "Bias"] <- coef_X - true_effect
}

cat("All regressions completed.\n")

In [ ]:
# Display and save results table
results_display <- results_modified
results_display$Beta <- round(results_display$Beta, 4)
results_display$SE <- round(results_display$SE, 4)
results_display$Bias <- round(results_display$Bias, 4)

cat("\nResults Table (Effect of X on Y):\n")
cat(paste(rep("=", 50), collapse = ""), "\n")
print(results_display)
cat(paste(rep("=", 50), collapse = ""), "\n")
cat(sprintf("True effect: %.3f\n", true_effect))

In [ ]:
# Save results table in different formats
# Save as CSV
write.csv(results_modified, file.path(output_dir, "part3_results_table_R.csv"), row.names = FALSE)

# Create a nicely formatted table for LaTeX
latex_table <- xtable(results_display, 
                     caption = "Results Table: Effect of X on Y",
                     label = "tab:part3_results")
print(latex_table, file = file.path(output_dir, "part3_results_table_R.tex"), 
      include.rownames = FALSE)

# Save as formatted text
sink(file.path(output_dir, "part3_results_table_R.txt"))
cat("Results Table: Effect of X on Y\n")
cat(paste(rep("=", 40), collapse = ""), "\n")
print(results_display, row.names = FALSE)
cat(sprintf("\n\nTrue causal effect: %.3f\n", true_effect))
sink()

cat("\nResults table saved in multiple formats:\n")
cat("- part3_results_table_R.csv\n")
cat("- part3_results_table_R.tex\n") 
cat("- part3_results_table_R.txt\n")

## Analysis and Conclusions

In [ ]:
# Identify which estimates are closest to the true effect
results_modified$abs_bias <- abs(results_modified$Bias)
best_estimates <- results_modified[order(results_modified$abs_bias), ][1:3, ]

cat("\nBest Estimates (lowest absolute bias):\n")
print(round(best_estimates[, c("Controls", "Beta", "SE", "Bias")], 4))

# Find minimal sufficient set
tolerance <- 0.05  # Consider estimates within 0.05 of true effect as "good"
good_estimates <- results_modified[results_modified$abs_bias < tolerance, ]

cat(sprintf("\nEstimates within %.2f of true effect:\n", tolerance))
if (nrow(good_estimates) > 0) {
  print(round(good_estimates[, c("Controls", "Beta", "SE", "Bias")], 4))
  
  # Find minimal set (fewest controls)
  good_estimates$num_controls <- ifelse(good_estimates$Controls == "None", 0, 
                                       lengths(strsplit(good_estimates$Controls, ", ")))
  minimal_idx <- which.min(good_estimates$num_controls)
  minimal_set <- good_estimates[minimal_idx, ]
  
  cat(sprintf("\nMinimal sufficient set: %s\n", minimal_set$Controls))
  cat(sprintf("Estimate: %.4f (SE: %.4f)\n", minimal_set$Beta, minimal_set$SE))
} else {
  cat("No estimates within tolerance found.\n")
}

## Answers to Questions

### In what way(s) can you get a good estimate of the causal effect?

Based on the results above, good estimates can be obtained by:

1. **Controlling for U1 and U2**: These block the backdoor paths through the confounders
2. **The combination that includes both U1 and U2** provides the most unbiased estimate
3. **Controlling for Z alone may not be sufficient** because Z is now on the causal path (mediator) and also a confounder

### What is the minimal sufficient set of controls?

The minimal sufficient set appears to be **{U1, U2}** because:
- U1 is a confounder (affects both X and Y)
- U2 is a confounder (affects Z which affects both X and Y)
- Controlling for both blocks all backdoor paths from X to Y

### Intuition for why these controls provide good estimates

The intuition is based on blocking backdoor paths:

1. **U1 → X ← Z → Y**: U1 confounds X and Y directly
2. **U2 → Z → X** and **U2 → Z → Y**: U2 confounds X and Y through Z
3. **By controlling for U1 and U2**, we block these confounding paths
4. **Z is problematic** because it's both a confounder and mediator - controlling for it blocks some confounding but also blocks part of the causal effect
5. **The backdoor criterion** suggests controlling for {U1, U2} satisfies the conditions for causal identification

In [ ]:
# Save all data
write.csv(df_modified, file.path(output_dir, "part3_modified_data_R.csv"), row.names = FALSE)

cat("\nAll files saved to output directory:\n")
cat("- part3_original_dag_R.png\n")
cat("- part3_original_coefficients_R.png\n")
cat("- part3_modified_dag_R.png\n")
cat("- part3_results_table_R.csv\n")
cat("- part3_results_table_R.tex\n")
cat("- part3_results_table_R.txt\n")
cat("- part3_modified_data_R.csv\n")
cat("\nPart 3 analysis complete (R implementation)!\n")